<a href="https://colab.research.google.com/github/we-human-centric/CursoPythonDatos_2026/blob/main/Dia%2015%20-%202026-05-13%20-%20APIs%20con%20FastAPI/clase_15.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Día 15 · APIs con FastAPI

Ayer hiciste un dashboard. Es una app web. Un usuario abre un navegador, ve gráficos, mueve sliders. Bonito.

Pero no todas las máquinas que necesitan tus datos tienen un navegador. Otro equipo en tu empresa quiere consultar tu modelo desde su aplicación Python. Una app móvil necesita predecir algo. Un workflow automatizado corre cada día. Para eso existe la API REST.

Hoy exponés tu modelo como endpoints HTTP. Cualquiera puede enviar un JSON, tu app devuelve una predicción. Sin saber Python. Sin descargar notebooks. Simplemente consumiendo una URL.

In [ ]:
# --- Setup del entorno
import random
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (10, 4)

print("Setup completo ✓")

## ¿Qué es una API REST?

Una **API** es un contrato entre dos piezas de software: "si me envías X, te devuelvo Y".

Una **API REST** (Representational State Transfer, definida por Roy Fielding en 2000) es el estilo dominante hoy en la web. Usa:

- **URLs** para identificar recursos: `/usuarios/42`, `/predicciones`.
- **Verbos HTTP**: GET (leer), POST (crear), PUT (actualizar), DELETE (borrar).
- **JSON** para los datos.
- **Códigos HTTP estandarizados**: 200 (ok), 404 (no encontrado), 500 (error del servidor).

En analítica, una API es potencia: **desacopla el modelo de quién lo consume**. Tu dashboard lo usa. Una app móvil lo usa. Un script automatizado lo usa. Todos sin saber qué hay adentro.

## Por qué FastAPI

FastAPI (Sebastián Ramírez, 2018) ganó popularidad porque combina tres cosas que ningún otro framework tenía:

1. **Velocidad** comparable a Go o Node — compite con Node.js en benchmarks.
2. **Validación automática** — declaras los tipos con type hints y FastAPI lo valida automáticamente.
3. **Documentación Swagger autogenerada** — tu API se auto-documenta en `/docs` con UI interactiva.

Es decir: escribís Python normal, FastAPI te da todo gratis. En 2023 era ya el framework más usado en la comunidad para servir modelos.

## Pydantic: validación basada en tipos

**Pydantic** valida datos automáticamente. Declares las estructuras así:

In [ ]:
from pydantic import BaseModel, Field

class Pasajero(BaseModel):
    """Esquema de entrada: lo que espera la API."""
    Age: float = Field(..., ge=0, le=120, description="Edad en años")
    Pclass: int = Field(..., ge=1, le=3, description="Clase (1, 2 o 3)")
    SexN: int = Field(..., ge=0, le=1, description="1=mujer, 0=hombre")
    Fare: float = Field(..., ge=0, description="Tarifa en dólares")

# Cuando alguien envía datos, Pydantic los valida automáticamente
# Si envían Age=150, Pydantic rechaza porque violó le=120
# Si envían Age="treinta", Pydantic rechaza porque no es float

print("Esquema definido. Pydantic lo valida.")

Sin Pydantic tendrías que validar manualmente cada campo. Con él, basta con declarar el tipo y las restricciones. Eso reduce código de boilerplate enormemente.

## De entrenamiento a producción

1. **Entrena el modelo** (pandas, scikit-learn).
2. **Serializalo** con `joblib.dump()` — lo guardás en un archivo `.joblib`.
3. **Cargalo en la API** al arrancar.
4. **Define el esquema** de entrada con Pydantic.
5. **Escribe el endpoint** que recibe datos, predice y devuelve.
6. **Probá** con `requests` o con Swagger UI.

## Entrenemos y guardemos un modelo

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
import joblib

# Cargar datos
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url).dropna(subset=["Age"]).copy()
df["SexN"] = (df["Sex"] == "female").astype(int)

# Seleccionar features
X = df[["Age", "Pclass", "SibSp", "Parch", "SexN", "Fare"]]
y = df["Survived"]

# Entrenar
Xtr, _, ytr, _ = train_test_split(X, y, test_size=0.2, random_state=SEED)
modelo = LogisticRegression(max_iter=1000, random_state=SEED).fit(Xtr, ytr)

# Guardar
joblib.dump(modelo, "modelo.joblib")
print(f"Modelo guardado. Accuracy en train: {modelo.score(Xtr, ytr):.3f}")

## Ahora la API

In [ ]:
CODIGO_API = '''
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
import joblib
import numpy as np

app = FastAPI(
    title="API Predicción Titanic",
    description="Predice si un pasajero habría sobrevivido",
    version="1.0.0",
)

# Cargar el modelo una sola vez al arrancar
modelo = joblib.load("modelo.joblib")

class Pasajero(BaseModel):
    Age: float = Field(..., ge=0, le=120)
    Pclass: int = Field(..., ge=1, le=3)
    SibSp: int = Field(0, ge=0)
    Parch: int = Field(0, ge=0)
    SexN: int = Field(..., ge=0, le=1)
    Fare: float = Field(..., ge=0)

class PrediccionOut(BaseModel):
    proba_sobrevivir: float
    prediccion: int

@app.get("/", tags=["meta"])
def root():
    return {"status": "ok", "modelo": "LogReg Titanic v1"}

@app.post("/predict", response_model=PrediccionOut, tags=["predicciones"])
def predecir(p: Pasajero):
    """Predice supervivencia dado un pasajero."""
    x = np.array([[p.Age, p.Pclass, p.SibSp, p.Parch, p.SexN, p.Fare]])
    try:
        proba = float(modelo.predict_proba(x)[0, 1])
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))
    return PrediccionOut(
        proba_sobrevivir=round(proba, 3),
        prediccion=int(proba > 0.5),
    )
'''
print(CODIGO_API)

Guardá ese código en `api.py`. Luego:

```bash
pip install fastapi uvicorn scikit-learn joblib
uvicorn api:app --reload
```

Abre dos URLs:
- `http://127.0.0.1:8000` → tu endpoint raíz.
- `http://127.0.0.1:8000/docs` → UI Swagger autogenerada. Ahí testeas la API en el navegador sin código.

## Pausa: ¿cómo Swagger sabe qué espera la API?

Porque FastAPI **lee tus type hints y docstrings** y genera automáticamente la documentación OpenAPI. Declaraste que `/predict` recibe un `Pasajero` y devuelve un `PrediccionOut`. FastAPI lo expone en Swagger sin que escribas una línea de documentación manual.

Eso es enorme en equipos grandes. Todo el mundo ve la misma documentación que el código genera. Nunca quedan desincronizadas.

## Consumir la API desde Python

In [ ]:
# Cuando la API está corriendo en localhost:8000 (descomenta para probar):

# import requests
# 
# payload = {
#     "Age": 29,
#     "Pclass": 1,
#     "SibSp": 0,
#     "Parch": 0,
#     "SexN": 1,
#     "Fare": 100.0,
# }
# 
# response = requests.post("http://127.0.0.1:8000/predict", json=payload)
# print(response.status_code)  # 200 = ok
# print(response.json())       # {"proba_sobrevivir": 0.925, "prediccion": 1}

print("✓ Descomenta para probar cuando la API esté corriendo")

## Consideraciones de producción

Lo que hicimos hoy es un MVP (Minimum Viable Product). Para ir a producción real necesitarías:

- **Autenticación**: API key, OAuth, tokens JWT.
- **Rate limiting**: evitar abuso (máximo 100 requests por minuto).
- **Logging estructurado**: registrar qué requests llegaron, cuáles fallaron, cuánto demoraron.
- **Monitoreo del modelo**: detectar data drift (cuando los datos de entrada cambian y el modelo se desactualiza).
- **Versionado**: si sacas un modelo v2, ¿qué pasa con los clientes que usan v1?
- **Contenedor Docker**: para despliegue reproducible.

Nada de esto es trivial. Pero todo parte de un endpoint bien diseñado como el que acabamos de crear.

## Checkpoint

1. **API REST** = interfaz HTTP para máquinas (GET, POST, JSON).
2. **FastAPI** = framework moderno que genera documentación automática.
3. **Pydantic** = validación de tipos automática.
4. **Serialización** = guardar el modelo con `joblib.dump()`.
5. **Swagger UI** en `/docs` para probar sin código.

Si algo quedó confuso, ahora es mejor aclarar.

In [ ]:
# Validación
assert 'CODIGO_API' in dir(), 'Deberías haber visto el código'
print("✓ Checkpoint completado")

## El flujo de una predicción

```
Cliente envía JSON con datos
       ↓
  FastAPI recibe POST /predict
       ↓
  Pydantic valida esquema
       ↓
  modelo.predict_proba() ejecuta
       ↓
  FastAPI devuelve JSON con predicción
       ↓
  Cliente parsea el JSON y actúa
```

## Patrones comunes

- **GET para leer**: `/modelo/info`, `/modelos` (lista).
- **POST para crear/predecir**: `/predict`, `/entrenar`.
- **Status codes legibles**: 200 (ok), 400 (bad request), 404 (no encontrado), 500 (error nuestro).
- **Errores con contexto**: no solo {"error": "fail"}. Sí: {"detail": "Age debe estar entre 0 y 120"}.
- **Batch predictions**: si es lento predecir uno a uno, permitir un POST con array de datos.

## Para tu equipo

Construyan una API con:
- Endpoint POST `/predict` que reciba features y devuelva predicción.
- Esquema Pydantic validado.
- Documentación Swagger en `/docs`.
- Modelo serializado y cargado al arrancar.

El entregable es `app/api.py` funcional, desplegable con `uvicorn api:app`.

---

Hoy desacoplaste tu modelo del cliente. Un usuario con Python puede llamar a tu API. Una app móvil puede llamar a tu API. Un workflow automatizado puede llamar a tu API. Todos ven el mismo modelo, la misma verdad, sin duplicar código ni saber Python.

Mañana vemos cómo empaquetar todo esto (tests, estructura, POO) para que un reclutador lo lea y piense "este persona sabe entregar software profesional", no solo "sabe entrenar modelos".